In [3]:
import json
import os
from datasets import Dataset
from transformers import AutoTokenizer
from modelscope import snapshot_download

def load_json_data(json_path):
    """加载JSON数据集"""
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"找不到文件：{json_path}")
        
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    # 过滤无效数据（确保每个样本有instruction和output）
    valid_data = [
        sample for sample in data 
        if "instruction" in sample 
        and "output" in sample
        and sample["instruction"].strip()
        and sample["output"].strip()
    ]
    print(f"加载数据：共{len(data)}条，有效数据{len(valid_data)}条")
    return valid_data

def format_deepseek_conversation(example):
    """
    按 DeepSeek 格式拼接对话
    将instruction作为用户输入，output作为助手回答
    """
    system_prompt = "现在你要扮演皇帝身边的女人--甄嬛"
    user_content = example["instruction"].strip()
    assistant_content = example["output"].strip()
    
    formatted_text = f"<|system|>{system_prompt}<|end|><|user|>{user_content}<|end|><|assistant|>{assistant_content}<|end|>"
    return {"text": formatted_text}

def prepare_dataset(json_path, model_name=snapshot_download('deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B')):
    """预处理入口：加载→格式化→分词"""
    # 1. 加载原始JSON数据
    raw_data = load_json_data(json_path)
    # 2. 转为 Hugging Face Dataset 格式（便于批量处理）
    dataset = Dataset.from_list(raw_data)
    # 3. 按模型格式拼接对话
    dataset = dataset.map(format_deepseek_conversation)
    # 4. 加载模型专属分词器
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,  # 加载自定义模型需开启
        padding_side="right"     # 右padding（避免生成时混乱）
    )
    # 补充模型必要的特殊token（若未自动识别）
    special_tokens = {
        "pad_token": "<|pad|>",
        "eos_token": "<|end|>",
        "additional_special_tokens": ["<|system|>", "<|user|>", "<|assistant|>"]
    }
    tokenizer.add_special_tokens(special_tokens)
    
    # 5. 分词（转为模型输入的张量）
    def tokenize_fn(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=384,  # 模型最大支持长度（可按需调整）
            padding="max_length",  # 统一填充到最大长度
            return_tensors="pt"
        )
    tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
    
    # 6. 划分训练集/验证集（9:1）
    split_dataset = tokenized_dataset.train_test_split(test_size=0.3, seed=42)
    return split_dataset["train"], split_dataset["test"], tokenizer

# ------------------- 测试数据预处理 -------------------
def test_dataset(json_path):
    """测试数据集处理流程"""
    try:
        # 1. 测试训练数据处理
        print(f"\n处理训练数据集：{json_path}")
        train_ds, val_ds, tokenizer = prepare_dataset(json_path)
        print(f"训练集：{len(train_ds)}条，验证集：{len(val_ds)}条")
        
        # 2. 测试未分词的数据格式
        print("\n验证数据格式：")
        raw_data = load_json_data(json_path)
        if raw_data:
            formatted_dataset = Dataset.from_list(raw_data).map(format_deepseek_conversation)
            
            # 显示前3条或更少的格式化文本
            num_samples = min(3, len(formatted_dataset))
            for i in range(num_samples):
                print(f"\n第{i+1}条格式化文本：")
                print(formatted_dataset[i]["text"])
        
    except Exception as e:
        print(f"处理过程中出现错误：{str(e)}")

# 运行测试（使用实际的数据文件）
test_dataset("huanhuan.json")

ModuleNotFoundError: No module named 'datasets'

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch
from modelscope import snapshot_download

def load_quantized_model(model_name=snapshot_download('deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B')):
    # 4-bit量化配置（大幅降低显存占用）
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    
    # 加载模型
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",  # 自动分配设备（GPU优先）
        trust_remote_code=True
    )
    
    # 启用梯度检查点（进一步节省显存）
    model.gradient_checkpointing_enable()
    return model

# 测试加载
if __name__ == "__main__":
    model = load_quantized_model()
    print(f"模型加载完成，设备：{model.device}")


2026-01-13 15:26:28,963 - modelscope - INFO - Target directory already exists, skipping creation.


模型加载完成，设备：cuda:0


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    TrainingArguments, Trainer, DataCollatorForLanguageModeling,
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
import torch
from modelscope import snapshot_download

# 1. 加载模型（动态探测结构）
def load_quantized_model(model_name=snapshot_download('deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B')):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    
    # 关键：动态打印模型的所有属性，找到存储Transformer层的主模块
    print("模型所有属性：", dir(model))  # 列出所有可用属性
    
    # 尝试常见的主模块名（根据输出结果手动选择）
    possible_main_modules = ["transformer", "model", "qwen", "deepseek"]
    main_module = None
    for name in possible_main_modules:
        if hasattr(model, name):
            main_module = name
            print(f"找到可能的主模块：{name}")
            # 打印该模块下的第一层结构
            layers = getattr(getattr(model, name), "layers", None) or getattr(getattr(model, name), "h", None)
            if layers is not None:
                print(f"主模块{name}的第一层结构：{layers[0]}")
                break
    
    if main_module is None:
        raise ValueError("未找到模型的主模块，请根据上面的属性列表手动指定")
    
    model = prepare_model_for_kbit_training(model)
    return model, main_module  # 返回主模块名，用于后续LoRA配置

# 2. 配置LoRA（根据动态获取的主模块调整）
def configure_lora(model, main_module):
    # 从主模块中提取实际的Transformer层，获取模块名
    layers = getattr(getattr(model, main_module), "layers", None) or getattr(getattr(model, main_module), "h", None)
    if layers is None:
        raise ValueError(f"主模块{main_module}中未找到layers或h属性")
    
    # 打印第一层的所有子模块，确认注意力/前馈层的名称
    print(f"第一层的子模块：{[name for name, _ in layers[0].named_modules() if '.' not in name]}")
    
    # 根据打印结果填写正确的模块名（以下为常见默认值，需根据实际输出修改）
    # target_modules = [
    #     "q_proj", "k_proj", "v_proj", "o_proj",  # 注意力层
    #     "gate_proj", "up_proj", "down_proj"       # 前馈层
    # ]

    target_modules = [
        "q_proj", "k_proj", "v_proj"  # 注意力层
        ]
    
    lora_config = LoraConfig(
        r=16,                            # LoRA秩
        lora_alpha=32,                  # 增加lora_alpha以提升性能
        target_modules=target_modules,  # 根据实际模块名调整
        lora_dropout=0.05,              # LoRA dropout
        bias="none",                    # 不微调偏置
        task_type="CAUSAL_LM",
        inference_mode=False
    )
    
    model = get_peft_model(model, lora_config)
    
    # 检查可训练参数
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"可训练参数: {trainable_params} | 总参数: {all_param} | 占比: {100 * trainable_params / all_param:.4f}%")
    
    if trainable_params == 0:
        raise ValueError("未找到可训练参数！请根据第一层子模块列表修改target_modules")
    
    return model

# 3. 训练配置（保持不变）
def train_model(model, train_dataset, val_dataset, tokenizer, output_dir="./deepseek_finetuned"):
    model.train()

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=4,    # 降低到最小批次
        per_device_eval_batch_size=1,     
        do_eval=False,                    
        eval_strategy="no",               
        num_train_epochs=3,               # 减少训练轮次到3轮
        learning_rate=1e-4,               # 略微提高学习率补偿小批次
        logging_steps=10,                 # 更频繁记录日志以监控训练
        save_strategy="steps",            # 按步数保存
        save_steps=500,                   # 降低保存频率，每500步保存一次
        fp16=True,                        
        gradient_accumulation_steps=4,    # 降低梯度累积步数
        gradient_checkpointing=True,      
        gradient_checkpointing_kwargs={"use_reentrant": False},  # 避免潜在的内存泄漏
        optim="paged_adamw_8bit",        
        lr_scheduler_type="cosine",       
        warmup_ratio=0.05,
        max_grad_norm=0.3,               # 添加梯度裁剪
        max_steps=-1,                    # 使用轮次控制而不是步数
        dataloader_num_workers=0,        # 减少数据加载器的进程数
        dataloader_pin_memory=False,     # 关闭内存钉扎
    )
    
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
        pad_to_multiple_of=8
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=lambda eval_pred: {
            "perplexity": torch.exp(torch.tensor(eval_pred.loss))
        }
    )
    
    trainer.train()
    model.save_pretrained(f"{output_dir}/best_lora_weights")
    print(f"最优模型保存至：{output_dir}/best_lora_weights")

# 主函数
if __name__ == "__main__":
    train_dataset, val_dataset, tokenizer = prepare_dataset("huanhuan.json")
    model, main_module = load_quantized_model()  # 获取模型和主模块名
    model = configure_lora(model, main_module)   # 传入主模块名配置LoRA
    train_model(model, train_dataset, val_dataset, tokenizer)

2026-01-13 16:17:45,076 - modelscope - INFO - Target directory already exists, skipping creation.


加载数据：共3729条，有效数据3729条


Map:   0%|          | 0/3729 [00:00<?, ? examples/s]

Map:   0%|          | 0/3729 [00:00<?, ? examples/s]

模型所有属性： ['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_assisted_decoding', '_auto_class', '_backward_compatibility_gradient_checkpointing', '_backward_hooks', '_backward_pre_hooks', '_beam_search', '_beam_search_has_unfinished_sequences', '_buffers', '_cache_dependant_input_preparation', '_cache_dependant_input_preparation_exporting', '_call_impl', '_can_compile_fullgraph', '_can_record_outputs', '_can_set_attn_implementation', '_check_and_adjust_attn_implementation', '_check_early_stop_heuristic', '_checkpoint_conversion_mapping', '_compiled_call_impl', '_constrained_beam_search', '_cont

Step,Training Loss
10,8.196000
20,8.029000
30,7.334100
40,6.772300
50,6.221700
60,5.777300
70,5.532600
80,5.440500
90,5.379400
100,5.351400


最优模型保存至：./deepseek_finetuned/best_lora_weights


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from modelscope import snapshot_download

def load_finetuned_model(
    base_model_id='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
    lora_path="./deepseek_finetuned/best_lora_weights"
):
    # 下载基础模型并获取本地路径
    base_model_path = snapshot_download(base_model_id)
    
    # 加载基础模型（同训练时的量化配置）
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_path,  # 使用本地路径
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    
    # 加载LoRA权重
    model = PeftModel.from_pretrained(base_model, lora_path)
    
    # 加载分词器
    tokenizer = AutoTokenizer.from_pretrained(base_model_path)
    tokenizer.pad_token = tokenizer.eos_token
    # 补充：设置<|end|>为终止token（与训练格式对齐）
    if "<|end|>" in tokenizer.get_vocab():
        tokenizer.eos_token_id = tokenizer.convert_tokens_to_ids("<|end|>")
    
    return model, tokenizer, base_model_id

def chat(model, tokenizer, system_prompt, user_input):
    """严格对齐训练格式：<|system|>xxx<|end|><|user|>xxx<|end|><|assistant|>xxx<|end|>"""
    # 1. 构建标准Prompt
    prompt = (
        f"<|system|>{system_prompt}<|end|>"
        f"<|user|>{user_input}<|end|>"
        f"<|assistant|>"
    )
    
    # 2. 编码输入
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=384  # 确保与训练时一致
    ).to(model.device)
    
    # 3. 生成回复
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,  # 解包输入张量
                max_new_tokens=80,  #生成最大长度
                min_new_tokens=5,  #最小生成长度
                temperature=0.5,  #控制生成多样性
                top_p=1.0,  #top-p采样
                do_sample=True,  #开启采样
                pad_token_id=tokenizer.pad_token_id,  # 设置填充token
                eos_token_id=tokenizer.eos_token_id,  # 设置终止token
                no_repeat_ngram_size=2,  #防止重复
                repetition_penalty=1.1  #惩罚重复
            )
    except Exception as e:
        raise RuntimeError(f"生成回复失败：{str(e)}") from e
    
    # 4. 提取并清理回复
    full_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=False,  # 保留特殊token以便截取
        clean_up_tokenization_spaces=True  # 清理空格
    )
    
    return full_text

# 测试对话（修复参数传递错误）
if __name__ == "__main__":
    try:
        model, tokenizer, model_id = load_finetuned_model()
        # 系统提示
        system_prompt = "现在你要扮演皇帝身边的女人--甄嬛"
        
        for i in range(3):
            user_input = "你是谁？"
            response = chat(model, tokenizer, system_prompt, user_input)
            print(f"用户：{user_input}")

            marker = "<|assistant|>"
            result = response.split(marker, 1)[1]  # 分割一次，取第二部分
            
            print(f"AI：{result}")

    except Exception as e:
        print(f"发生错误：{str(e)}")


2026-01-13 20:25:01,667 - modelscope - INFO - Target directory already exists, skipping creation.


用户：你是谁？
AI：我是皇帝的侧耳，您是何人？请自我介绍一下。<||end||>```<｜end▁of▁sentence｜>
用户：你是谁？
AI：我是臣妾，甄娘娘。您是怎样的人？臣服于您，您是我的母亲。臣属是臣臣，臣奴是贵妃。我是贵嫔的臣子。贵王是您的父方。你可记得了？是的，贵宫里有太多事要办，我得先去见贵主。是啊，你也是。我
用户：你是谁？
AI：我是华贵宫。<|\end><｜end▁of▁sentence｜>
